In [ ]:
function range(n: number): number[] {
    return Array.from({ length: n + 1 }, (_, i) => i);
}

In [ ]:
range(4)

# Computing the Conjunctive Normal Form in First Order Logic

In order to convert a formula $f$ from first order logic into a set of clauses that is satisfiable if and only if $f$ is satisfiable,
we have to perform the following steps in order:
- eliminate biconditionals 

  Here is it is important to rename bound variables in order to avoid clashes,
- eliminate conditionals,
- transform the formula into *negation normal form*,
  i.e. we push the negation symbol inwards,

- transform the formula into *prenex normal form*,
  i.e. we move the quantifieres outside,
  
- eliminate existential quantifiers by *skolemizing* the formula, and
- transform the formula into *clauses* in set notation.

When converting formulas into conjunctive normal form, we <u>assume</u> that the formulas are 
*pure*, where we define a formula $f$ as *pure* if all quantifiers appearing in $f$ bind **different** variables.  For example, the formula
$$ \bigl(\forall X: p(X)\bigr) \vee \bigl(\forall X: q(X)\bigr)$$
is **not** *pure*, because there are two different universal quantifiers that both bind the same variable $X$.  We can rewrite this formulas as a *pure* formula by *renaming* all occurrences of $X$ that are bound by the second quantifier as follows:
$$ \bigl(\forall X: p(X)\bigr) \vee \bigl(\forall Y: q(Y)\bigr)$$

## Auxilliary Functions

Formulas are represented as nested tuples.  In order to convert a string into a nested tuple we use the parser that is found in the module `FOL-Parser`.  Our parser distinguishes variables and function symbol as follows:
- A word starting with an *upper* case letter is interpreted as a *variable*.
- A word starting with a *lower* case letter is assumed to be a *function* or *predicate symbol*.

In [ ]:
import { parseFormula as parse, Formula, Term, Variable, PredicateSymbol, FunctionSymbol } from "./FOL-Parser";
import { Tuple, RecursiveSet as Set, Value, flatMap } from "recursive-set";

In [ ]:
function set<T extends Value>(...elements: T[]): Set<T> {
    return new Set(...elements);
}

In [ ]:
function tpl<T extends Value[]>(...elements: T): Tuple<T> {
    return new Tuple(...elements);
}

For testing purposes, the following formula is used.  This formula specifies the notion of a *grandparent*.

In [ ]:
const s = '∀G:∀C:(grandparent(G, C) ↔ ∃P: (parent(G, P) ∧ parent(P, C)))';
const f1 = parse(s);
console.dir(f1, { depth: null });

A `Substitution` maps variables to terms. 

In [ ]:
type Substitution = Map<Variable, Term>;

The function $\texttt{applyTerm}(t, σ)$ takes a term $t$ and a *variable substitution* 
$\sigma = \{ x_1 \mapsto s_1, \cdots, x_n \mapsto s_n\}$ which is represented as a `Map` and replaces every occurrence of the variable $x_i$ in the object $f$ with the corresponding term $s_i$.

In [ ]:
function applyTerm(t: Term, sigma: Substitution): Term {
    if (typeof t === 'string') {
        const mapped = sigma.get(t);
        return mapped !== undefined ? mapped : t;
    }
    const [f, ...args] = t;
    return [f, ...args.map(arg => applyTerm(arg, sigma))];
}

The function $\texttt{applyFormula}(t, σ)$ takes a Formula $t$ and a *variable substitution* 
$\sigma = \{ x_1 \mapsto s_1, \cdots, x_n \mapsto s_n\}$ which is represented as a `Map` and replaces every occurrence of the variable $x_i$ in the formula $f$ with the corresponding term $s_i$.

In [ ]:
function applyFormula(f: Formula, sigma: Substitution): Formula {
    switch (f[0]) {
        case '⚛️': {
            const [tag, pred, ...args] = f;
            return [tag, pred, ...args.map(arg => applyTerm(arg, sigma))];
        }
        case '⊤':
        case '⊥':
            return f;
        case '¬': {
            const [op, g] = f;
            return [op, applyFormula(g, sigma)];
        }
        case '∧':
        case '∨':
        case '→':
        case '↔': {
            const [op, g, h] = f;
            return [op, applyFormula(g, sigma), applyFormula(h, sigma)];
        }
        case '∀':
        case '∃': {
            const [op, x, g] = f;
            const mapped = sigma.get(x);
            const newX = typeof mapped === 'string' ? mapped : x;
            return [op, newX, applyFormula(g, sigma)];
        }
    }
}

In [ ]:
console.dir(f1, { depth: null });

In [ ]:
const sigma1: Substitution = new Map([
    ['G', 'X'],
    ['P', 'Y'],
    ['C', 'Z']
]);
console.dir(applyFormula(f1, sigma1), { depth: null });

The function $\texttt{boundVariables}(f)$ computes the set of variables that are *bound* in the formula $f$. 

In [ ]:
function boundVariables(f: Formula): Set<string> {
    switch (f[0]) {
        case '⚛️':
        case '⊤':
        case '⊥':
            return set<string>();
        case '¬': {
            const [_, g] = f;
            return boundVariables(g);
        }
        case '∧':
        case '∨':
        case '→':
        case '↔': {
            const [_, g, h] = f;
            return boundVariables(g).union(boundVariables(h));
        }
        case '∀':
        case '∃': {
            const [_, x, g] = f;
            return boundVariables(g).union(set(x));
        }
    }
}

In [ ]:
console.dir(f1, { depth: null });

In [ ]:
console.log([...boundVariables(f1)]);

The function `allVariablesTerm` computes the set of all variables that occur in the given term.

In [ ]:
function allVariablesTerm(t: Term): Set<string> {
    if (typeof t === 'string') return set(t);
    const [_, ...args] = t;
    return flatMap(args, arg => allVariablesTerm(arg));
}

The function `allVariables` computes the set of all variables that occur in the given formula.

In [ ]:
function allVariables(f: Formula): Set<string> {
    switch(f[0]) {
        case '⚛️': {
            const [_, pred, ...args] = f;
            return flatMap(args, arg => allVariablesTerm(arg));
        }
        case '⊤':
        case '⊥':
            return set<string>();
        case '¬': {
            const [_, g] = f;
            return allVariables(g);
        }
        case '∧':
        case '∨':
        case '→':
        case '↔': {
            const [_, g, h] = f;
            return allVariables(g).union(allVariables(h));
        }
        case '∀':
        case '∃': {
            const [_, x, g] = f;
            return allVariables(g).union(set(x));
        }
    }
}

In [ ]:
console.dir(f1, { depth: null });

In [ ]:
console.log([...allVariables(f1)]);

In [ ]:
const g1: Formula = ['↔', 
    ['⚛️', 'grandparent', 'G', 'C'],
    ['∃', 'P', ['∧', ['⚛️', 'parent', 'G', 'P'], ['⚛️', 'parent', 'P', 'C']]]
];

In [ ]:
console.log([...allVariables(g1)]);

Below we construct a list of all upper case characters to generate new variables.

In [ ]:
const ascii_uppercase = "ABCDEFGHIJKLMNOPQRSTUVWXYZ".split("");
ascii_uppercase

In [ ]:
const ascii_set = set(...ascii_uppercase);
console.log(ascii_set.size);

The function $\texttt{renameBoundVariables}(f)$ takes a first order formula $f$ and replaces all bound variables by **new** variables.  This only works if the set of characters `set(string.ascii_uppercase)` has enough characters that do not already occur in $f$.  This approach would not be good enough for a production quality program,
but for the case of a demonstration it is sufficient.  The alternative would be to rename the variables as `X1`, `X2`, `X3`, $\cdots$, but that becomes unreadable very fast.

In [ ]:
function renameBoundVariables(f: Formula): Formula {
    const boundVs = [...boundVariables(f)];
    const allVs   = allVariables(f);
    const newVars = ascii_uppercase.filter(x => !allVs.has(x)).sort();
    const mappingPairs = boundVs.map((bv, i) => [bv, newVars[i]] as const);
    const sigma: Substitution = new Map(mappingPairs);    
    return applyFormula(f, sigma);
}

In [ ]:
console.log(['A', 'B', 'C'].map((x, i) => [i, x]));

In [ ]:
console.dir(f1, { depth: null });

In [ ]:
console.dir(renameBoundVariables(f1), { depth: null });

## Elimination Biconditionals

The function $\texttt{eliminateBiconditional}(f)$ takes a formula $f$ from first order logic and eliminates all occurrences of the operator '↔' from this formula.  This is done by using the following equivalence:
$$(f \leftrightarrow g) \;\Leftrightarrow\; (f \rightarrow g) \wedge (g \rightarrow f)$$
In order to ensure that the resulting formula is <em style="color:blue">pure</em>, we have to rename the bound variables in the formula $g \rightarrow f$.

In [ ]:
function eliminateBiconditional(f: Formula): Formula {
    switch (f[0]) {
        case '⚛️':
        case '⊤':
        case '⊥':
            return f;
        case '¬': {
            const [op, g] = f;
            return [op, eliminateBiconditional(g)];
        }
        case '∧':
        case '∨':
        case '→': {
            const [op, g, h] = f;
            return [op, eliminateBiconditional(g), eliminateBiconditional(h)];
        }
        case '↔': {
            const [_, g, h] = f;
            const ge = eliminateBiconditional(g);
            const he = eliminateBiconditional(h);
            const left: Formula = ['→', ge, he];
            const right = renameBoundVariables(['→', he, ge]);
            return ['∧', left, right];
        }
        case '∀':
        case '∃': {
            const [op, x, g] = f;
            return [op, x, eliminateBiconditional(g)];
        }
    }
}

In [ ]:
console.dir(f1, { depth: null });

In [ ]:
const f2 = eliminateBiconditional(f1);
console.dir(f2, { depth: null });

## Eliminating Conditionals

The function $\texttt{eliminateConditional}(f)$ takes a formula $f$ from first order logic and eliminates all occurrences of the operator '→' from this formula.  This is done by using the following equivalence:
$$(f \rightarrow g) \;\Leftrightarrow\; (\neg f \vee g)$$
The implementation of this function is similar to the implementation of the function `eliminateConditional` that we had used in propositional logic.

In [ ]:
function eliminateConditional(f: Formula): Formula {
    switch (f[0]) {
        case '⚛️':
        case '⊤':
        case '⊥':
            return f;
        case '¬': {
            const [op, g] = f;
            return [op, eliminateConditional(g)];
        }
        case '∧':
        case '∨':
        case '↔': {
            const [op, g, h] = f;
            return [op, eliminateConditional(g), eliminateConditional(h)];
        }
        case '→': {
            const [_, g, h] = f;
            return ['∨', ['¬', eliminateConditional(g)], eliminateConditional(h)];
        }
        case '∀':
        case '∃': {
            const [op, x, g] = f;
            return [op, x, eliminateConditional(g)];
        }
    }
}

In [ ]:
console.dir(f2, { depth: null });

In [ ]:
const f3 = eliminateConditional(f2);
console.dir(f3, { depth: null });

## Negation Normal Form

The function $\texttt{nnf}(f)$ computes the <em style="color:blue;">negation normal form</em> of $f$, while $\texttt{neg}(f)$ computes the *negation normal form* of $\neg f$. 

The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all propositional variables $p$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2). 
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(\forall x: F \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg \forall x: F\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\exists x: \neg F\bigr) \\[0.1cm]
       = & \exists x: \texttt{nnf}(\neg F)           \\[0.1cm]
       = & \exists x: \texttt{neg}(F).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(\forall x: F \bigr) = \exists x: \texttt{neg}(F)$.</li>
      <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(\exists x: F \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg \exists x: F\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\forall x: \neg F\bigr) \\[0.1cm]
       = & \forall x: \texttt{nnf}(\neg F)           \\[0.1cm]
       = & \forall x: \texttt{neg}(F).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(\exists x: F \bigr) = \forall x: \texttt{neg}(F)$.</li>
</ol>

In [ ]:
function nnf(f: Formula): Formula {
    switch (f[0]) {
        case '⚛️':
        case '⊤':
        case '⊥':
            return f;
        case '¬': {
            const [_, g] = f;
            return neg(g);
        }
        case '∧':
        case '∨':
        case '→':
        case '↔': {
            const [op, g, h] = f;
            return [op, nnf(g), nnf(h)];
        }
        case '∀':
        case '∃': {
            const [op, x, g] = f;
            return [op, x, nnf(g)];
        }
    }
}

function neg(f: Formula): Formula {
    switch (f[0]) {
        case '⊤': return ['⊥'];
        case '⊥': return ['⊤'];
        case '¬': {
            const [_, g] = f;
            return nnf(g);
        }
        case '∧': {
            const [_, g, h] = f;
            return ['∨', neg(g), neg(h)];
        }
        case '∨': {
            const [_, g, h] = f;
            return ['∧', neg(g), neg(h)];
        }
        case '→':
        case '↔':
        case '⚛️':
            return ['¬', f];
        case '∀': {
            const [_, x, g] = f;
            return ['∃', x, neg(g)];
        }
        case '∃': {
            const [_, x, g] = f;
            return ['∀', x, neg(g)];
        }
    }
}

In [ ]:
console.dir(f3, { depth: null });

In [ ]:
const f4 = nnf(f3);
console.dir(f4, { depth: null });

## Prenex Normal Form

In the following we assume that all quantifiers that occur in a formula bind **different** variables, i.e. we assume that the formulas are *pure*.  If this assumption is not satisfied, then the functions given below will produce <u>garbage</u>.

In [ ]:
type Quantifier = '∀' | '∃';
type QuantifierPrefix = [Quantifier, Variable];
type QuantifierList = QuantifierPrefix[];

A *quantifier tuple* is a tuple of the following form:
$$ (Q_1, x_1, \cdots, Q_n, x_n) $$
Here, the $Q_i$ denote quantifiers, i.e. we have $Q_i \in \{\forall, \exists\}$, while the $x_i$ are variables.  The function $\texttt{mergeQuantifiers}(T_1, T_2)$ takes two quantifier tuples $T_1$ and $T_2$ as arguments and merges them into a new quantifier tuple such that the relative order of the quantifiers remains the same, i.e. if both $Q_1, x_1$ and $Q_2, x_2$ occur in $T_1$ and $Q_1, x_1$ occurs before $Q_2, x_2$, then $Q_1, x_1$ will occur before $Q_2, x_2$ in the result.

In [ ]:
function mergeQuantifiers(Q1: QuantifierList, Q2: QuantifierList): QuantifierList {
    if (Q1.length === 0) return Q2;
    if (Q2.length === 0) return Q1;
    if (Q1[0][0] === '∃') return [Q1[0], ...mergeQuantifiers(Q1.slice(1), Q2)];
    if (Q2[0][0] === '∃') return [Q2[0], ...mergeQuantifiers(Q1, Q2.slice(1))];
    return [Q1[0], ...mergeQuantifiers(Q1.slice(1), Q2)];
}

In [ ]:
console.log(mergeQuantifiers([['∀', 'X'], ['∃', 'Y']], [['∃', 'U'], ['∀', 'V']]));

Given a formula $f$, the function $\texttt{extractQuantifiers}(f)$ returns a pairs $(T, m)$, where $T$ is a quantifier tuple and $m$ is the <em style="color:blue;">matrix</em> of the formula $f$, where the matrix of a formula is defined as the part that remains when all quantifiers have been extracted.

In [ ]:
function extractQuantifiers(f: Formula): [QuantifierList, Formula] {
    switch (f[0]) {
        case '⚛️':
        case '⊤':
        case '⊥':
        case '¬':
            return [[], f];
        case '∧':
        case '∨':
        case '→':
        case '↔': {
            const [op, g, h] = f;
            const [qg, gm] = extractQuantifiers(g);
            const [qh, hm] = extractQuantifiers(h);
            return [mergeQuantifiers(qg, qh), [op, gm, hm]];
        }
        case '∀':
        case '∃': {
            const [op, x, g] = f;
            const [qg, gm] = extractQuantifiers(g);
            return [[[op, x], ...qg], gm];
        }
    }
}

In [ ]:
console.dir(f4, { depth: null });

In [ ]:
const [Qs, f5] = extractQuantifiers(f4);
console.log(Qs);
console.dir(f5, { depth: null });

Given a qantifier tuple $\texttt{Qs}$ and a matrix $m$, the call $\texttt{attachQuantifiers}(Qs, m)$ combines the quantifiers $\texttt{Qs}$ and the matrix $m$ into a quantified formula.

In [ ]:
function attachQuantifiers(Qs: QuantifierList, m: Formula): Formula {
    if (Qs.length === 0) return m;
    const [Q, x] = Qs[0];
    return [Q, x, attachQuantifiers(Qs.slice(1), m)];
}

In [ ]:
console.log(Qs);

In [ ]:
console.dir(f5, { depth: null });

In [ ]:
const f6 = attachQuantifiers(Qs, f5);
console.dir(f6, { depth: null });

## Skolemization (Eliminating Existential Quantifiers)

The variable $\texttt{skolemCounter}$ is a global variable that is needed to create unique Skolem constants.  

In [ ]:
let skolemCounter = 0;

In [ ]:
function skolemConstant(): string {
    skolemCounter += 1;
    return 'sk' + skolemCounter.toString();
}

The function $\texttt{skolemize}(f, \texttt{Vs})$ takes a formula $f$ and a tuple of variables $\texttt{Vs}$ and 
<em style="color:blue">skolemizes</em> the formula $f$, i.e. it replaces all existentially quantified variables by appropriate <em style="color:blue">Skolem functions</em>.  The tuple $\texttt{Vs}$ is a tuple of variables that are 
assumed to be universally quantified.  The formula $f$ is assumed to be in <em style="color:blue">prenex normal form</em>.

For skolemization to work correctly, we have to assume that 
<font size="4" style="color:darkgreen; size:125%">$f$ does not contain free variables</font>!

In [ ]:
function skolemize(f: Formula, Vs: string[]): Formula {
    switch (f[0]) {
        case '∃': {
            const [_, x, g] = f;
            const t: Term = [skolemConstant(), ...Vs];
            const sigma: Substitution = new Map();
            sigma.set(x, t);
            return skolemize(applyFormula(g, sigma), Vs);
        }
        case '∀': {
            const [op, x, g] = f;
            return [op, x, skolemize(g, [...Vs, x])];
        }
        default:
            return f;
    }
}

In [ ]:
console.dir(f6, { depth: null });

In [ ]:
const f7 = skolemize(f6, []);
console.dir(f7, { depth: null });

## Conversion to Clauses

In [ ]:
type TupleTerm = string | Tuple<[FunctionSymbol, ...TupleTerm[]]>;
type TupleAtom = Tuple<['⚛️', PredicateSymbol, ...TupleTerm[]]>;

type Literal = TupleAtom | Tuple<['¬', TupleAtom]>;
type Clause = Set<Literal>;
type CNFSet = Set<Clause>;

function termToTuple(t: Term): TupleTerm {
    if (typeof t == 'string') return t;
    const [f, ...args] = t;
    return tpl(f, ...args.map(termToTuple));
}

function literalToTuple(f: Formula): Literal {
    if (f[0] == '⚛️') {
        const [tag, pred, ...args] = f;
        return tpl(tag, pred, ...args.map(termToTuple));
    }
    if (f[0] == '¬') {
        const [_, g] = f;
        if (g[0] == '⚛️') {
            const [_, pred, ...args] = g;
            return tpl('¬', tpl('⚛️', pred, ...args.map(termToTuple)));
        }
    }
}

The function $\texttt{cnf}(f)$ takes a <em style="color:blue">skolemized</em> formula $f$ from first order logic that is in <em style="color:blue">negation normal form</em> and returns the <em style="color:blue">conjunctive normal form</em> of $f$ in <em style="color:blue">set notation</em>.  This works the same way as in propositional logic.

In [ ]:
function cnf(f: Formula): CNFSet {
    switch (f[0]) {
        case '⊤': 
            return set<Clause>();
        case '⊥': 
            return set(set<Literal>());
        case '¬': 
        case '⚛️':
            return set(set(literalToTuple(f)));
        case '∧': {
            const [_, g, h] = f;
            return cnf(g).union(cnf(h));
        }
        case '∨': {
            const [_, g, h] = f;
            return flatMap(cnf(g), k1 => cnf(h).map(k2 => k1.union(k2)));
        }
        case '∀': {
            const [_, x, g] = f;
            return cnf(g);
        }
        default:
            throw new Error(`Unexpected operator in CNF conversion: ${f[0]}`);
    }
}

In [ ]:
console.dir(f7, { depth: null });

In [ ]:
const f8 = cnf(f7);
for (let cl of f8) {
    console.log(cl.toString());
}

## Putting Everything Together

The function $f$ takes a <em style="color:blue">pure</em> formula $f$ from first order logic and transforms $f$ into a set of first order clauses.  Furthermore, $f$ **must not** contain free variables.

In [ ]:
function normalize(f: Formula): CNFSet {
    const f1 = eliminateBiconditional(f);
    const f2 = eliminateConditional(f1);
    const f3 = nnf(f2);
    const [Qs, f4] = extractQuantifiers(f3);
    const f5 = attachQuantifiers(Qs, f4);
    const f6 = skolemize(f5, []);
    return cnf(f6);
}

In [ ]:
for (let cl of normalize(f1)) {
    console.log(cl.toString());
}

In [ ]:
function prettify(M: CNFSet): string {
    if (M.size === 0) return '{}';
    let result = "{\n";
    const clauses = [...M];
    for (let i = 0; i < clauses.length; i++) {
        const A = clauses[i];
        if (A.size === 0) {
            result += "    {},\n";
        } else {
            result += "    {" + [...A].map(lit => lit.toString()).join(", ") + "}";
            if (i < clauses.length - 1) result += ",\n";
            else result += "\n";
        }
    }
    result += "}";
    return result;
}

In [ ]:
function test(s: string): void {
    const f = parse(s);
    console.log(`The knf of ${s} is:`);
    console.log(prettify(normalize(f)));
}

In [ ]:
test(s);

In [ ]:
test('¬(∃Y:∀X:p(X,Y)→∀U:∃V:p(U,V))');